# 从 fMRI beta 到脑活动向量：逐单元格教学

按顺序阅读说明与紧接着的代码、输出。只读取现有数据，不下载模型、不训练、不生成单独结果文件。

**主线**：行为表确定 trial → 读取官方 beta → 抽取顶点 → 每 session 标准化 → 拼接 trial → 过滤无效顶点 → 按图像 ID 平均重复 → 转置成脑特征矩阵。

这里得到的是**脑活动特征向量**，不是语言模型语义向量。最后解释两者如何通过监督学习连接，但不执行模型。

每个代码单元都可以在已有 `visuo_llm` 内核执行；修改参数后必须从头重跑，避免旧变量混用。

## 01｜先理解 beta 从哪里来（概念，不重新处理原始 BOLD）

输入：原始 fMRI 的 BOLD 时间序列，以及每张图像的呈现时间。BOLD 反映血氧变化，不是直接记录神经元放电。

上游做法：时间/空间等预处理、对齐到共同皮层空间；用刺激时序和血流动力学响应函数 HRF 构建设计矩阵，以 GLM 估计每次呈现的响应幅度 beta。本数据目录名 `betas_fithrf_GLMdenoise_RR` 表示拟合 HRF、GLMdenoise 去噪、ridge 正则化的官方估计；去噪减少无关波动，正则化缓解相近 trial 回归变量相关带来的估计不稳定。

输出：每个皮层顶点、每个 trial 的一个 beta。本教程已经从这个输出开始，**没有原始 BOLD，不重跑上述步骤**。beta 不是每一秒的时间点，也不是概率。

依据：[GLMsingle 官方说明](https://github.com/cvnlab/GLMsingle/blob/main/docs/source/wiki.md)。

In [ ]:
print("本次实际起点：官方已处理 MGH beta；不是 raw BOLD。")
print("本次实际终点：每张教学图像对应一个脑活动向量。")
print("不会运行：GLM/GLMdenoise、语义模型、回归训练。")

## 02｜路径、参数与原仓库函数

输入：Docker 中已有仓库和数据。做法：导入科学计算包，直接加载原项目的工具模块。为什么：真实调用原函数，同时不触发完整实验入口的训练或文件保存。

输出：数据路径、抽样参数、函数名。教学抽样不是解剖学 ROI，不用于复现论文全脑数值。

In [ ]:
import sys, os, json, hashlib, importlib.util, io, contextlib
from pathlib import Path
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
import nibabel as nib
from scipy.stats import zscore
from PIL import Image

REPO = Path("/workspace/projects/visuo_llm")
DATA = Path(os.environ.get("NSD_TEACHING_DATA_DIR", "/workspace/datasets/nsd_teaching_b"))
SUBJECT = "subj01"
SESSIONS = list(range(1, 7))
VERTICES_PER_HEMISPHERE = 512
assert VERTICES_PER_HEMISPHERE > 0
source = REPO / "src/nsd_visuo_semantics/utils/nsd_get_data_light.py"
spec = importlib.util.spec_from_file_location("teaching_nsd_utils", source)
utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(utils)
np.set_printoptions(precision=4, suppress=True)
print("Python:", sys.executable)
print("数据:", DATA)
print("原函数文件:", source.relative_to(REPO))
print("直接调用: read_behavior, get_conditions, get_subject_conditions, average_over_conditions")
print("教学展开: get_betas 的 fsaverage 分支（先抽样，减少内存）")

## 03｜读取固定清单：不能重新随机选样本

输入：已有 `data_manifest.json`。做法：读取已下载的 200 张图像 ID、caption、文件信息。为什么：之后必须用同一套 ID 对齐脑响应与图像；Git 同步代码不等于同步数据。

输出：清单哈希、数量和样本记录。此处不重新下载、不写清单。

In [ ]:
manifest_path = DATA / "data_manifest.json"
manifest_bytes = manifest_path.read_bytes()
manifest = json.loads(manifest_bytes)
assert manifest["subject"] == SUBJECT and manifest["sessions"] == SESSIONS
selected = manifest["selected"]
image_ids = np.array(sorted(x["nsd_id"] for x in selected), dtype=np.int64)
assert len(image_ids) == len(np.unique(image_ids)) == 200
by_id = {x["nsd_id"]: x for x in selected}
print("清单 SHA256:", hashlib.sha256(manifest_bytes).hexdigest())
print("图像/完整PNG:", len(image_ids), "caption:", sum(len(x["captions"]) for x in selected))
print("源文件数:", len(manifest["files"]))
print("首个样本:", {k: selected[0][k] for k in ("nsd_id", "caption_index", "trial_count", "image_key")})

## 04｜行为表：每个 beta 列究竟对应什么

输入：`nsddata/ppdata/subj01/behav/responses.tsv`。做法：调用 `read_behavior(..., session_index=1)` 按 SESSION 筛选。为什么：beta 列没有自动附上图像文字，必须从行为表取 `73KID`。

输出：session 1 的 750 行及列名。注意原函数 docstring 写“从 0 计数”，但实际过滤 SESSION，本数据实际使用 **1–6**。保持表中顺序，不按图像 ID 对 trial 排序。

In [ ]:
behavior1 = utils.read_behavior(str(DATA), SUBJECT, 1)
print("session 1 行列:", behavior1.shape)
print("列名:", list(behavior1.columns))
print(behavior1.head(5).to_string(index=False))
assert len(behavior1) == 750
print("前 8 个图像 ID:", behavior1["73KID"].to_numpy()[:8])

## 05｜建立 4500 个 trial 的图像标签

输入：六个 session 的行为表。做法：调用原 `get_conditions` 并按 session 顺序拼接。为什么：稍后脑数据必须按完全相同顺序拼接。

输出：一维 `conditions[trial]`；同一 ID 多次出现表示图像重复呈现，不是多张不同图像。

In [ ]:
with contextlib.redirect_stdout(io.StringIO()):
    conditions_by_session = utils.get_conditions(str(DATA), SUBJECT, len(SESSIONS))
conditions = np.concatenate(conditions_by_session).astype(np.int64)
assert [len(x) for x in conditions_by_session] == [750] * 6
trial_table = pd.DataFrame({
    "global_trial_0based": np.arange(len(conditions)),
    "session": np.repeat(SESSIONS, 750),
    "within_session_0based": np.tile(np.arange(750), 6),
    "nsd_id": conditions,
})
print("conditions:", conditions.shape, conditions.dtype)
print(trial_table.iloc[[0, 1, 749, 750, 4499]].to_string(index=False))
print("注意：trial 下标从 0 开始；NSD ID 从 1 开始。")

## 06｜打开 MGH：读文件头，不加载全脑

输入：session 1 左半球的 MGH。做法：`nibabel.load` 返回图像及延迟读取代理 `dataobj`。为什么：先看形状，避免原 `get_fdata()` 一次读出完整 float64 全脑数组。

输出：`(163842, 1, 1, 750)`；第一轴为 fsaverage 皮层顶点，最后轴为 trial。此处叫“顶点”不是体积空间的“体素”。

In [ ]:
beta_dir = DATA / "nsddata_betas/ppdata" / SUBJECT / "fsaverage/betas_fithrf_GLMdenoise_RR"
def beta_path(hemisphere, session):
    return beta_dir / f"{hemisphere}.betas_session{session:02d}.mgh"

lh_img = nib.load(beta_path("lh", 1))
rh_img = nib.load(beta_path("rh", 1))
assert lh_img.shape == rh_img.shape
N_VERTICES, one1, one2, N_TRIALS = lh_img.shape
assert (one1, one2, N_TRIALS) == (1, 1, 750)
print("左半球:", lh_img.shape, "存储 dtype:", lh_img.get_data_dtype())
print("右半球:", rh_img.shape)
print("代理类型:", type(lh_img.dataobj).__name__)
print("全两半球顶点数:", 2 * N_VERTICES)

## 07｜取一个小数据块：看清 squeeze 的含义

输入：左半球前 8 个顶点、前 6 个 trial。做法：先切片读取，再 `squeeze` 去除长度为 1 的空间轴。为什么：保留“顶点 × trial”二维布局；不是改变 beta 数值，不是 AI 编码。

输出：`(8,1,1,6) → (8,6)`。可能包含无效顶点，稍后统一过滤。

In [ ]:
small4d = np.asarray(lh_img.dataobj[:8, :, :, :6], dtype=np.float64)
small2d = small4d.squeeze(axis=(1, 2))
print("squeeze 前/后:", small4d.shape, "→", small2d.shape)
print("前 8 顶点 × 前 6 trial 的原始 beta:")
print(small2d)

## 08｜固定抽样顶点：降低教学内存

输入：每半球 163842 个顶点。做法：固定步长抽样，每半球最多 512 个。为什么：先学习轴、索引与数值变换，不必一次读完整全脑。

输出：所有 session 使用完全相同的顶点索引。原 `get_betas` 是先逐顶点 zscore 再 mask；这里先选顶点再逐顶点 zscore，因各行标准化独立，对保留行的结果等价。这个抽样不意味着选中了视觉 ROI。

In [ ]:
stride = max(1, int(np.ceil(N_VERTICES / VERTICES_PER_HEMISPHERE)))
vertex_indices = np.arange(0, N_VERTICES, stride)
global_vertex_ids = np.concatenate([vertex_indices, N_VERTICES + vertex_indices])
print("步长:", stride, "实际每半球:", len(vertex_indices))
print("半球内索引前 8:", vertex_indices[:8])
print("全局索引：左半球保持原索引，右半球加", N_VERTICES)
print("两半球抽样总特征数:", len(global_vertex_ids))

## 09｜读取 session 1 的全部 trial

输入：左右半球 MGH 与抽样索引。做法：代理按固定步长读顶点、读取全部 750 trial，再变成二维。为什么：标准化基准要覆盖整个 session，不能只读教学图像 trial 就当作原方法。

输出：左/右各 `(抽样顶点数,750)` float64，与原 `get_fdata` 工作精度一致。

In [ ]:
def read_sampled(hemisphere, session):
    img = nib.load(beta_path(hemisphere, session))
    assert img.shape == (N_VERTICES, 1, 1, N_TRIALS)
    return np.asarray(img.dataobj[::stride, :, :, :], dtype=np.float64).squeeze(axis=(1, 2))

lh_raw = read_sampled("lh", 1)
rh_raw = read_sampled("rh", 1)
print("左:", lh_raw.shape, lh_raw.dtype, "右:", rh_raw.shape, rh_raw.dtype)
print("第一条非零且有限左半球行（前 6 trial）:")
valid_lh = np.isfinite(lh_raw).all(axis=1) & (np.std(lh_raw, axis=1) > 0)
assert valid_lh.any()
print(lh_raw[np.flatnonzero(valid_lh)[0], :6])

## 10｜合并左右半球：增加特征，不增加 trial

输入：左右两个“顶点 × trial”矩阵。做法：`np.vstack` 沿顶点轴拼接。为什么：同一个 trial 需要同时包含左右半球响应。

输出：`(左顶点+右顶点,750)`；前半行是 lh，后半行是 rh，最后轴仍与行为表对应。

In [ ]:
session1_raw = np.vstack([lh_raw, rh_raw])
print("vstack:", lh_raw.shape, "+", rh_raw.shape, "→", session1_raw.shape)
v = int(np.flatnonzero(np.isfinite(session1_raw).all(axis=1) & (np.std(session1_raw, axis=1) > 0))[0])
print("用于手算的抽样行:", v, "原始全局顶点 ID:", global_vertex_ids[v])
print("该顶点前 6 个 trial:", session1_raw[v, :6])

## 11｜手算一个顶点的 session z-score

输入：一个顶点在 750 个 trial 上的 beta。做法：`z=(beta−该行均值)/该行标准差`，SciPy 默认 `ddof=0`。为什么：减小 session 间基线与尺度差别，保留该 session 内相对响应。

输出：相对响应，可负可正；不是概率。标准差为零会产生无效值，不能强行当作有信息的向量。标准化会去掉这一级的绝对幅度信息。

In [ ]:
mu = session1_raw[v].mean()
sigma = session1_raw[v].std(ddof=0)
manual_z = (session1_raw[v] - mu) / sigma
print("750 trial 均值:", mu, "标准差:", sigma)
print(pd.DataFrame({"beta": session1_raw[v,:6], "beta-mu": session1_raw[v,:6]-mu,
                    "z": manual_z[:6]}).to_string(index=False))
print("标准化后整行均值/标准差:", manual_z.mean(), manual_z.std())

## 12｜对所有抽样顶点按 trial 轴标准化

输入：session 1 的二维 beta。做法：原方法 `zscore(axis=-1)`，之后 `astype(float32)`。为什么：每个顶点单独处理；最后轴是 trial，不能误对顶点轴标准化。

输出：形状不变、数值尺度改变、内存精度变为 float32。fsaverage 分支**不除以 300**；原体积空间 func1pt8mm 分支才有该步骤。

In [ ]:
def standardize_session(raw):
    with np.errstate(divide="ignore", invalid="ignore"):
        return zscore(raw, axis=-1).astype(np.float32)

session1_z = standardize_session(session1_raw)
assert np.allclose(session1_z[v], manual_z, atol=1e-6)
print("shape:", session1_z.shape, "dtype:", session1_z.dtype)
print("手算与 SciPy 一致:", np.allclose(session1_z[v], manual_z, atol=1e-6))
print("该行前 6 个 z:", session1_z[v,:6])
print("含无效值的顶点行数:", int((~np.isfinite(session1_z).all(axis=1)).sum()))

## 13｜逐 session 处理：不要跨 session 算一个均值

输入：六对 MGH。做法：各 session 独立读取、左右合并、标准化。为什么：原 `get_betas` 的循环就是按 session 建立独立标准化基准。

输出：六个同形矩阵。这里只用抽样顶点，不调用会加载全脑的 `get_betas`，也不调用会保存 npy 的 `load_or_compute_betas_average`。

In [ ]:
raw_sessions = [session1_raw]
z_sessions = [session1_z]
for session in SESSIONS[1:]:
    raw = np.vstack([read_sampled("lh", session), read_sampled("rh", session)])
    raw_sessions.append(raw)
    z_sessions.append(standardize_session(raw))
summary = [{"session": s, "shape": str(z.shape), "dtype": str(z.dtype),
            "invalid_vertex_rows": int((~np.isfinite(z).all(axis=1)).sum())}
           for s,z in zip(SESSIONS,z_sessions)]
print(pd.DataFrame(summary).to_string(index=False))

## 14｜拼接 session：这次增加 trial

输入：六个“顶点 × 750 trial”矩阵。做法：`np.concatenate(axis=-1)`。为什么：保持顶点作为同一特征，沿 trial 轴依次连接；与步骤 05 的 conditions 顺序一致。

输出：`(抽样顶点数,4500)`。session 1 最后列与 session 2 第一列的边界可核验。

In [ ]:
all_z = np.concatenate(z_sessions, axis=-1)
all_raw = np.concatenate(raw_sessions, axis=-1)
assert all_z.shape[1] == len(conditions)
assert np.array_equal(all_z[:,750], z_sessions[1][:,0], equal_nan=True)
print("脑矩阵:", all_z.shape, "标签:", conditions.shape)
print("边界两列图像ID:", conditions[749:751])
print("选定顶点的边界数值:", all_z[v,749:751])
print("抽样原始+标准化矩阵内存 MB:", (all_raw.nbytes+all_z.nbytes)/1e6)

## 15｜过滤无效顶点：过滤行而非 trial

输入：标准化矩阵。做法：仅保留在全部 4500 trial 都有限的行，保存对应全局顶点 ID。为什么：常数/缺失行不能作为可靠特征，NaN/Inf 会使后续模型失败。这里用严格有限性过滤；不能把缺失值悄悄变成零。

输出：trial 数不变、特征数可能减少。这是教学的显式质控，保留“列特征来自哪个顶点”的索引。

In [ ]:
valid_vertices = np.isfinite(all_z).all(axis=1)
brain_trials = all_z[valid_vertices,:]
kept_vertex_ids = global_vertex_ids[valid_vertices]
assert brain_trials.shape[0] > 0
assert np.isfinite(brain_trials).all()
print("过滤前:", all_z.shape, "→ 后:", brain_trials.shape)
print("删除顶点行数:", int((~valid_vertices).sum()))
print("保留全局顶点 ID 前 10:", kept_vertex_ids[:10])
print("不是删除 trial：列数仍为", brain_trials.shape[1])

## 16｜选择 200 张图像的 trial：过滤列

输入：4500 个 trial 标签和固定 200 个教学 ID。做法：`np.isin` 建立列掩码。为什么：下载的是 200 张完整图像，但 beta 文件含整个 session；现在取这些图像真正对应的响应。

输出：349 个呈现 trial；重复次数可能为 1、2、3。每个 ID 与清单记录对照。

In [ ]:
trial_mask = np.isin(conditions, image_ids)
teaching_trials = brain_trials[:,trial_mask]
teaching_conditions = conditions[trial_mask]
repeat_counts = np.array([np.sum(conditions == i) for i in image_ids])
assert np.all(repeat_counts > 0)
assert all(int(c)==by_id[int(i)]["trial_count"] for i,c in zip(image_ids,repeat_counts))
print("所选 trial:", teaching_trials.shape, "标签:", teaching_conditions.shape)
print("图像数/次数分布:", len(image_ids), dict(zip(*np.unique(repeat_counts,return_counts=True))))
print("次数总和:", repeat_counts.sum())
print(trial_table.loc[trial_mask].head(8).to_string(index=False))

## 17｜手算同一图像重复呈现的平均

输入：一张出现三次的图像及其三列响应。做法：沿重复 trial 求均值。为什么：把“每次呈现的响应”变成“每张图像的较稳定响应”，减少不一致噪声；不能保证消除噪声，也会丢失重复间差异。

输出：一个长度等于有效顶点数的向量。没有三次的图像可平均现有一/两次，但可靠性可能不同。

In [ ]:
example_id = int(image_ids[np.flatnonzero(repeat_counts == 3)[0]])
example_cols = np.flatnonzero(conditions == example_id)
repeated = brain_trials[:,example_cols]
manual_mean = repeated.mean(axis=1)
print("示例图像 ID:", example_id, "全局 trial 下标:", example_cols)
print("重复矩阵:", repeated.shape, "平均向量:", manual_mean.shape)
print(pd.DataFrame({"repeat1":repeated[:6,0], "repeat2":repeated[:6,1],
                    "repeat3":repeated[:6,2], "mean":manual_mean[:6]}).to_string(index=False))

## 18｜调用原函数平均全部教学图像

输入：`teaching_trials`、逐 trial 的标签、200 个唯一 ID。做法：调用原 `average_over_conditions`。为什么：真正复用作者的条件平均逻辑，而不是用示意随机数组替代。

输出：`(有效顶点,200)`；原函数 `np.unique` 会按 ID 排序，`np.empty` 默认使输出为 float64。先验证所有 ID 存在，避免原函数遇到缺失 ID 时 break 留下未初始化区域。

In [ ]:
assert set(image_ids).issubset(set(teaching_conditions))
averaged = utils.average_over_conditions(teaching_trials, teaching_conditions, image_ids)
j = int(np.flatnonzero(image_ids == example_id)[0])
assert np.allclose(averaged[:,j], manual_mean, atol=1e-6)
assert averaged.shape == (len(kept_vertex_ids),len(image_ids))
print("原函数输出:", averaged.shape, averaged.dtype)
print("三次重复手算核对:", np.allclose(averaged[:,j],manual_mean,atol=1e-6))
print("列顺序：NSD ID 升序:", image_ids[:8])

## 19｜转置：每张图像一行，终于得到特征向量

输入：“顶点 × 图像”矩阵。做法：转置为“图像 × 特征”，转换 float32。为什么：机器学习常用 X 的每行作为一个样本，每列为一个特征；转置只是轴交换，不是学习。

输出：`X_brain.shape=(200,V)`；一行形状 `(V,)`。第 i 行对应 `image_ids[i]`，第 k 列对应 `kept_vertex_ids[k]`。这些顶点响应本身就是脑特征，**不是768维语义 embedding**。

In [ ]:
X_brain = averaged.T.astype(np.float32)
assert np.isfinite(X_brain).all()
brain_vector = X_brain[j]
print("最终 X_brain:", X_brain.shape, X_brain.dtype)
print("一条脑向量:", brain_vector.shape, "对应图像:", image_ids[j])
print("前 12 个特征:", brain_vector[:12])
print("总体 min/mean/max:", float(X_brain.min()),float(X_brain.mean()),float(X_brain.max()))
print("最终矩阵内存 KB:", X_brain.nbytes/1000)

## 20｜验证脑向量与真实图像/caption 对齐

输入：示例行的 ID 和清单。做法：读取对应完整 PNG，打印全部 caption；不编码文本。为什么：检查没有把某图像的脑响应配到另一张图像。

输出：425×425 RGB 图像的数组形状与五句描述。caption_index=NSD ID−1；这不是 trial 下标。原 caption 表路径见清单。

In [ ]:
record = by_id[example_id]
with Image.open(DATA / record["image_key"]) as im:
    pixels = np.array(im.convert("RGB"))
print("脑向量行:", j, "NSD ID:", example_id, "caption 表行:", record["caption_index"])
print("完整图像路径:", record["image_key"])
print("图像像素数组:", pixels.shape, pixels.dtype)
for n, sentence in enumerate(record["captions"],1):
    print(f"caption {n}: {sentence}")
assert record["caption_index"] == example_id - 1

## 21｜参数实验 A：标准化开/关，及先筛选的影响

输入：真实 beta。做法：比较不标准化的重复均值与原 session z-score 均值；再对单顶点比较“完整750trial算基准”与“只用教学trial算基准”。为什么：看到相同 shape 不代表相同意义。

输出：尺度/数值变化。**主流程仍沿用原每 session 标准化**。未来训练评估需检查标准化是否涉及测试数据；本教程不划分训练集、不报告泛化指标。

In [ ]:
raw_valid = all_raw[valid_vertices,:]
no_z_vector = raw_valid[:,example_cols].mean(axis=1)
print(pd.DataFrame({"no_z_raw_beta":no_z_vector[:6],
                    "session_z_mean":brain_vector[:6]}).to_string(index=False))
mask1 = np.isin(conditions_by_session[0], image_ids)
assert mask1.sum() > 1
subset_z = zscore(session1_raw[v,mask1])
original_z_selected = session1_z[v,mask1]
print("session1 教学trial数:", int(mask1.sum()), "/750")
print("标准化基准不同，前 6 值:")
print(pd.DataFrame({"all750_baseline":original_z_selected[:6],
                    "selected_only_baseline":subset_z[:6]}).to_string(index=False))

## 22｜参数实验 B：特征数与平均方法

输入：最终矩阵及三次重复。做法：选择已有有效列的前 64、256 或全部；比较单次、均值、中位数。为什么：顶点数量改变特征维度；重复汇总规则改变数值而不是特征维度。

输出：真实数组 shape/数值对照。前缀选列不是新 ROI，也不等于重新按不同步长采样。median 是教学备选，原方法是 mean。

In [ ]:
for k in sorted(set([min(64,X_brain.shape[1]), min(256,X_brain.shape[1]), X_brain.shape[1]])):
    print(f"保留前 {k} 个有效特征:", X_brain[:,:k].shape)
print(pd.DataFrame({"first_repeat":repeated[:6,0],
                    "mean_original":repeated[:6].mean(axis=1),
                    "median_alternative":np.median(repeated[:6],axis=1)}).to_string(index=False))
print("顶点抽样参数 VERTICES_PER_HEMISPHERE 改动后要从头重跑；不能混用旧索引。")

## 23｜参数实验 C：只保留三次重复会怎样

输入：六个 session 的行为表。做法：实际调用原 `get_subject_conditions` 的 False/True 两种设置，与200个教学 ID 取交集。为什么：原论文完整数据常筛三次重复；六session教学子集不能假设200张都三次。

输出：200张→36张，特征列不变。注意 True 判断的是**当前读取session中的出现次数恰好为3**，不是全实验是否最终出现3次。

In [ ]:
with contextlib.redirect_stdout(io.StringIO()):
    all_conditions, _, sample_all = utils.get_subject_conditions(str(DATA), SUBJECT,6,False)
    three_conditions, _, sample_three = utils.get_subject_conditions(str(DATA), SUBJECT,6,True)
assert np.array_equal(all_conditions,conditions) and np.array_equal(three_conditions,conditions)
three_ids = np.intersect1d(image_ids,sample_three)
three_rows = np.isin(image_ids,three_ids)
assert np.array_equal(three_ids,image_ids[repeat_counts==3])
print("False：教学图像矩阵", X_brain.shape)
print("True：教学图像矩阵", X_brain[three_rows].shape)
print("三个重复图像ID前 8:", three_ids[:8])
print("原函数样本池（未限制教学200张）:",len(sample_all),"→",len(sample_three))

## 24｜脑向量如何变成语义向量（只说明，不运行）

输入：本教程的 X_brain；另需每张图像对应的语言目标 Y。通常 caption → tokenizer（文字变 token ID）→ 已训练 Transformer → pooling/模型配置的归一化 → 每句语义向量 → 五句平均成每图像向量。模型权重来自预训练，不能只装 Python 包就拥有模型。

映射做法：先按同一 NSD ID 对齐 X/Y，再划分训练/验证/测试；监督回归用训练集学习 W 和截距 b，推理为 X_test @ W + b。原 `src/nsd_visuo_semantics/encoding_decoding_analyses/nsd_decode_llm.py::nsd_decode_llm` 使用第三方 `FracRidgeRegressorCV.fit/predict`。该映射是脑→语义“解码”；反方向语义→脑是“编码”。

输出：若目标维度 d=768，预测矩阵为 (测试图像数,768)。**这里没有 Y、没有已训练 W，因此没有语义预测，更不能把脑向量直接改名为语义向量。**

不用下载模型的选项：TF-IDF 可产生词频特征，但不是预训练语义模型、不可当作论文同一实验；PCA 可压缩脑特征，但不是语言语义空间。本次不运行这些替代方法，以免偏离脑数据主线。

In [ ]:
d_example = 768
print("已有输入 X_brain:", X_brain.shape)
print("尚未计算 Y_caption:", (len(image_ids), d_example), "（仅示意形状）")
print("未训练 W:", (X_brain.shape[1],d_example), "未训练截距 b:",(d_example,))
print("仅示意，不执行：regressor.fit(X_train,Y_train); Y_pred=regressor.predict(X_test)")
print("原工具：get_embeddings/embedding_models_zoo.py::get_embedding_model / get_embeddings")
print("原 caption 汇总：encoding_decoding_analyses/encoding_decoding_utils.py::make_subj_conditional_nsd_embeddings")

## 25｜平时如何使用模型？需要多大？（本次不执行）

通常 `SentenceTransformer("sentence-transformers/all-mpnet-base-v2")` 会在本地缓存不存在时下载模型，然后 `model.encode(sentences)`。也能先下载完整配置、tokenizer、权重，再传本地路径；本地路径本身不会替代权重。网络 API 不需要本地权重，但需服务、网络、可能付费及数据传输授权，本次不调用。

| 可选模型 | 每句输出维数 | 单个 safetensors 权重 | 最小配置+tokenizer+权重的大致空间 |
| --- | --- | --- | --- |
| all-MiniLM-L6-v2 | 384 | 90.9 MB | 约100 MB |
| all-mpnet-base-v2 | 768 | 438 MB | 约450 MB |

这是十进制 MB 的估算，不是整个模型仓库（其中可能含重复格式/ONNX权重）的大小。来源：[MiniLM 模型卡](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)、[MiniLM 权重](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/blob/main/model.safetensors)、[MPNet 模型卡](https://huggingface.co/sentence-transformers/all-mpnet-base-v2)、[MPNet 权重](https://huggingface.co/sentence-transformers/all-mpnet-base-v2/blob/main/model.safetensors)。

本教程只需要 CPU 科学计算包，不需要以上模型、不需要 GPU；没有改变项目核心算法或安装依赖。

## 26｜最终核验与阅读路线

输入：全部真实中间结果。做法：检查有限值、标签顺序、形状、重复平均的一致性。为什么：只看到代码没报错还不够，要确认每一步保持了正确语义。

输出：全流程核验通过后，可在同一 notebook 查看每个输出；不另保存报告、npy、过程文件。

In [ ]:
assert len(conditions)==4500
assert teaching_trials.shape[1]==349
assert X_brain.shape==(200,len(kept_vertex_ids))
assert len(three_ids)==36 and np.isfinite(X_brain).all()
assert np.all(np.diff(image_ids)>0)
assert np.allclose(X_brain[j],manual_mean,atol=1e-6)
print("全部断言通过。")
print("beta→抽样→左右合并→每session z-score→跨session拼接→有限顶点过滤")
print("→按图像筛trial→重复平均→转置→X_brain",X_brain.shape)
print("阅读路线：README → 本notebook → nsd_get_data_light.py四个已调用函数")
print("→ get_betas → average_over_conditions → nsd_decode_llm（最后再读训练/预测）。")
print("本次不下载模型、不训练、不单独保存结果。")